In [1]:
import numpy as np
import pandas as pd
import math
import sqlite3
from   ucimlrepo import fetch_ucirepo
from   sklearn.model_selection import train_test_split
from   itertools import chain, combinations
from   more_itertools import powerset


path = 'wine.csv'
wine = pd.read_csv(path)


In [2]:
X = wine.drop(columns=['quality'])
y = wine['quality']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_mean   = X_train.mean()
X_centre = X_train - X_mean
X_cov    = X_train.cov()
X_cov_inv = np.linalg.inv(X_cov)



In [6]:
start_id, end_id       = 4656, 3659
start_point, end_point = X_test.loc[start_id], X_test.loc[end_id]
X_explain = pd.concat([X_train, start_point.to_frame().T, end_point.to_frame().T])

In [7]:
dist_M = np.matmul(X_cov_inv, X_explain.T)

In [2]:
check_table = '''SELECT EXISTS (SELECT name FROM sqlite_schema WHERE  type='table' AND  name='X_train');'''

conn = sqlite3.connect('wine.db')
cursor = conn.cursor()
if cursor.execute(check_table).fetchone()[0] == 0:
    wine_quality = fetch_ucirepo(id=186)['data']['original']
    wine_quality.insert(0, "id", range(1, len(wine_quality) + 1))
    X_train, X_test = train_test_split(wine_quality, test_size=0.2, random_state=42)
    X_train.to_sql('X_train', conn, if_exists='replace', index=True)
    X_test.to_sql('X_test', conn, if_exists='replace', index=True)

In [3]:
player_names = ['fixed_acidity', 'volatile_acidity', 'citric_acid','residual_sugar', 'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density', 'pH', 'sulphates', 'alcohol', 'quality']

target_player = 'quality'

player_set = set(player_names) - {target_player}

path, dist = dict(), dict()

start_id, end_id = 1087, 1062

for player_name in player_names:

    get_var   = f'''select avg(power(({player_name} - (select avg({player_name}) from X_train)),2)) from X_train'''
    get_value = f'''select {player_name} from X_test where id = :unique_id'''

    var       = cursor.execute(get_var).fetchone()[0]
    start_val = cursor.execute(get_value, {'unique_id': start_id}).fetchone()[0]
    end_val   =  cursor.execute(get_value, {'unique_id': end_id}).fetchone()[0]

    path[player_name] = [start_val, end_val]
    dist[player_name+'_stddev'] = math.sqrt(var)

    print(player_name.rjust(20),
           round( math.sqrt(var),3),
          str(start_val).ljust(6),
          str(end_val).ljust(6),
          str(round((end_val- start_val)/math.sqrt(var),6)).ljust(11))


       fixed_acidity 1.288 11.8   7.1    -3.648662  
    volatile_acidity 0.162 0.23   0.26   0.184997   
         citric_acid 0.145 0.38   0.49   0.760351   
      residual_sugar 4.796 11.1   2.2    -1.855666  
           chlorides 0.035 0.034  0.032  -0.057428  
 free_sulfur_dioxide 17.438 15.0   31.0   0.917543   
total_sulfur_dioxide 56.137 123.0  113.0  -0.178137  
             density 0.003 0.9997 0.9903 -3.124624  
                  pH 0.16 2.93   3.37   2.756677   
           sulphates 0.149 0.55   0.42   -0.874558  
             alcohol 1.191 9.7    12.9   2.686233   
             quality 0.877 3      9      6.844911   


In [4]:
def value(coalition):

    if len(coalition)==0:
        target = {player_name: path[player_name][0]  for player_name in player_names }
    else:
        target = {player_name: path[player_name][player_name in coalition]  for player_name in player_names }

    params = target | dist
    params['start_id'] = start_id
    params['end_id']   = end_id

    query = f'''select quality
                from   X_explain
                order by    power((fixed_acidity - :fixed_acidity)/:fixed_acidity_stddev ,2)
                          + power((volatile_acidity - :volatile_acidity)/:volatile_acidity_stddev ,2)
                          + power((citric_acid - :citric_acid)/:citric_acid_stddev ,2)
                          + power((residual_sugar - :residual_sugar)/:residual_sugar_stddev ,2)
                          + power((chlorides - :chlorides)/:chlorides_stddev ,2)
                          + power((free_sulfur_dioxide - :free_sulfur_dioxide)/:free_sulfur_dioxide_stddev ,2)
                          + power((total_sulfur_dioxide - :total_sulfur_dioxide)/:total_sulfur_dioxide ,2)
                          + power((density - :density)/ :density_stddev ,2)
                          + power((pH - :pH)/:pH_stddev,2)
                          + power((sulphates - :sulphates)/:sulphates_stddev,2)
                          + power((alcohol - :alcohol)/:alcohol_stddev,2) '''

    return cursor.execute(query, params).fetchone()[0]




In [14]:
def explain(conn, start_id, end_id):

    def gamma(players, coalition):
        N = len(players)
        S = len(coalition)
        return math.factorial(S) * math.factorial(N - S - 1) / math.factorial(N)

    phi = dict()

    cursor = conn.cursor()

    params = dict()
    params['start_id'] = start_id
    params['end_id']   = end_id

    X_explain = '''create table X_explain as
                   select * from X_train
                   union
                   select * from X_test where  id in (:start_id, :end_id)'''

    X_drop    = '''drop table X_explain'''

    cursor.execute(X_explain, params)

    for player in player_set:

        get_value = f'''select {player} from X_test where id = :unique_id'''
        start_val = cursor.execute(get_value, {'unique_id': start_id}).fetchone()[0]
        end_val   =  cursor.execute(get_value, {'unique_id': end_id}).fetchone()[0]

        get_var   = f'''select avg(power(({player} - (select avg({player_name}) from X_train)),2)) from X_train'''
        var       = cursor.execute(get_var).fetchone()[0]
        std_dev   = math.sqrt(var)

        d_M       = (end_val - start_val) / std_dev

        coalitions = player_set - {player}
        phi[player]=0.0
        for S in powerset(coalitions):
            v2 = value(set(S).union({player}))
            v1 = value(set(S))
            phi[player] += gamma(player_set, S) * (v2 - v1)

        print(player, start_val, end_val, round(std_dev,3), round(d_M,3), round(phi[player],3))

        cursor.execute(X_drop)

    return phi

explain(conn, start_id, end_id)


volatile_acidity 0.23 0.26 5.48 0.005 -0.042
chlorides 0.034 0.032 5.759 -0.0 0.002
free_sulfur_dioxide 15.0 31.0 30.277 0.528 0.351
total_sulfur_dioxide 123.0 113.0 123.406 -0.081 -0.005
density 0.9997 0.9903 4.82 -0.002 1.568
pH 2.93 3.37 2.602 0.169 1.291
alcohol 9.7 12.9 4.819 0.664 0.972
sulphates 0.55 0.42 5.286 -0.025 -0.009
residual_sugar 11.1 2.2 4.808 -1.851 0.147
fixed_acidity 11.8 7.1 1.894 -2.481 1.373
citric_acid 0.38 0.49 5.496 0.02 0.349


{'volatile_acidity': -0.04188311688311687,
 'chlorides': 0.0021645021645021645,
 'free_sulfur_dioxide': 0.35137085137085194,
 'total_sulfur_dioxide': -0.0045815295815295805,
 'density': 1.5684343434343426,
 'pH': 1.2906565656565656,
 'alcohol': 0.9724025974025975,
 'sulphates': -0.008549783549783484,
 'residual_sugar': 0.14740259740259734,
 'fixed_acidity': 1.3731962481962487,
 'citric_acid': 0.34938672438672447}

In [8]:
conn.close()

In [6]:
playr

NameError: name 'params' is not defined